In [43]:
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import joblib
from sklearn.preprocessing import LabelEncoder

In [59]:
salary_data = pd.read_csv('archive\salaries.csv')
print(salary_data)

       work_year experience_level employment_type                 job_title  \
0           2025               MI              FT  Customer Success Manager   
1           2025               SE              FT                  Engineer   
2           2025               SE              FT                  Engineer   
3           2025               SE              FT         Applied Scientist   
4           2025               SE              FT         Applied Scientist   
...          ...              ...             ...                       ...   
88579       2020               SE              FT            Data Scientist   
88580       2021               MI              FT  Principal Data Scientist   
88581       2020               EN              FT            Data Scientist   
88582       2020               EN              CT     Business Data Analyst   
88583       2021               SE              FT            Data Scientist   

        salary salary_currency  salary_in_usd emplo

<>:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
C:\Users\91981\AppData\Local\Temp\ipykernel_53532\1680199410.py:1: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  salary_data = pd.read_csv('archive\salaries.csv')


In [45]:
salary_data.isna().sum()

work_year             0
experience_level      0
employment_type       0
job_title             0
salary                0
salary_currency       0
salary_in_usd         0
employee_residence    0
remote_ratio          0
company_location      0
company_size          0
dtype: int64

In [46]:
salary_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 88584 entries, 0 to 88583
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   work_year           88584 non-null  int64
 1   experience_level    88584 non-null  str  
 2   employment_type     88584 non-null  str  
 3   job_title           88584 non-null  str  
 4   salary              88584 non-null  int64
 5   salary_currency     88584 non-null  str  
 6   salary_in_usd       88584 non-null  int64
 7   employee_residence  88584 non-null  str  
 8   remote_ratio        88584 non-null  int64
 9   company_location    88584 non-null  str  
 10  company_size        88584 non-null  str  
dtypes: int64(4), str(7)
memory usage: 7.4 MB


In [47]:
salary_data.describe()

,work_year,salary,salary_in_usd,remote_ratio
count,88584.000000,8.858400e+04,88584.000000,88584.000000
mean,2024.034758,1.619323e+05,157567.798417,21.286011
std,0.620370,1.965317e+05,73531.373158,40.831018
min,2020.000000,1.400000e+04,15000.000000,0.000000
25%,2024.000000,1.060000e+05,106097.250000,0.000000
50%,2024.000000,1.470000e+05,146307.000000,0.000000
75%,2024.000000,1.995000e+05,198600.000000,0.000000
max,2025.000000,3.040000e+07,800000.000000,100.000000


In [48]:
# Remove raw salary columns before encoding
salary_data = salary_data.drop(['salary', 'salary_currency'], axis=1)

In [49]:
print(salary_data['experience_level'].unique())
print(salary_data['employment_type'].unique())
print(salary_data['company_size'].unique())
print(salary_data['work_year'].unique())
print(salary_data['remote_ratio'].unique())

<StringArray>
['MI', 'SE', 'EN', 'EX']
Length: 4, dtype: str
<StringArray>
['FT', 'CT', 'PT', 'FL']
Length: 4, dtype: str
<StringArray>
['L', 'M', 'S']
Length: 3, dtype: str
[2025 2024 2022 2023 2020 2021]
[ 50   0 100]


In [50]:

y = salary_data['salary_in_usd'].values
top_10_title = salary_data['job_title'].value_counts().index[:10].tolist()
salary_data['job_title'] = salary_data['job_title'].apply(lambda x: x if x in top_10_title else 'Other')



categorical_cols = ['experience_level', 'employment_type', 'job_title', 'employee_residence', 'company_location', 'company_size']


encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    salary_data[col] = le.fit_transform(salary_data[col].astype(str))
    encoders[col] = le

In [51]:

X = salary_data.drop('salary_in_usd', axis=1).values

In [52]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [53]:
xg_reg = xgb.XGBRegressor(n_estimators=200, objective='reg:squarederror', random_state=42, max_depth=10, learning_rate=0.1)
xg_reg.fit(X_train, y_train)
preds = xg_reg.predict(X_test)
rmse = mean_squared_error(preds,y_test) ** 0.5
print (rmse)
print(salary_data['salary_in_usd'].mean())

64741.23162251395
157567.7984173214


In [54]:
filename = "salarypredictor.pkl"
joblib.dump(xg_reg,filename)

['salarypredictor.pkl']

In [55]:
print(X.shape)

(88584, 8)
